In [1]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().parent
train_path = PROJECT_ROOT / "all_datasets" / "breast_cancer_dataset" / "data.csv"
df = pd.read_csv(train_path)

In [3]:
X = df.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y = (df["diagnosis"] == "M").astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [4]:
# Select k only inside the training partition.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = [1, 3, 5, 7, 11, 21]
cv_scores = {}

for k in k_values:
    candidate = Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k, weights="distance")),
    ])
    scores = cross_val_score(candidate, X_train, y_train, cv=cv, scoring="roc_auc")
    cv_scores[k] = scores.mean()
    print(f"k={k:2d} mean training-CV ROC AUC={scores.mean():.3f}")

best_k = max(cv_scores, key=cv_scores.get)
print("selected k:", best_k)

k= 1 mean training-CV ROC AUC=0.946
k= 3 mean training-CV ROC AUC=0.984
k= 5 mean training-CV ROC AUC=0.985
k= 7 mean training-CV ROC AUC=0.987
k=11 mean training-CV ROC AUC=0.988
k=21 mean training-CV ROC AUC=0.988
selected k: 11


In [5]:
models = {
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=best_k, weights="distance")),
    ]),
    "GaussianNB": Pipeline([
        ("scale", StandardScaler()),
        ("model", GaussianNB()),
    ]),
}

In [6]:
# Validation compares the locked KNN choice with the alternative model family.
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    prob = model.predict_proba(X_valid)[:, 1]
    print(
        name,
        "validation accuracy=", round(accuracy_score(y_valid, pred), 3),
        "validation ROC AUC=", round(roc_auc_score(y_valid, prob), 3),
    )

KNN validation accuracy= 0.958 validation ROC AUC= 0.986
GaussianNB validation accuracy= 0.944 validation ROC AUC= 0.992
